# Two beams and a joint (compas_timber)

Companion to [Week 06 §11](../README.md#11-graphs-in-architecture-three-applications).
`compas_timber` models a timber structure as a **graph**: beams are nodes,
joints are edges. Before a whole wall, this notebook looks at the smallest
possible model: two beams, and the one decision between them.

> **Needs the timber extra.** Start Jupyter with
> `uv run --extra timber jupyter lab`. Everything here runs without Rhino;
> Step 6 shows where that stops.

> **Units are millimetres**, as in timber fabrication.

In [ ]:
# Setup: run this cell first. It finds the shared drawing helper (viz.py),
# whether this notebook sits in Lecture/notebooks/ or is your copy in MyWork/.
import sys
import pathlib

HERE = pathlib.Path.cwd()
NB = next(p / "Lecture" / "notebooks" for p in [HERE, *HERE.parents]
          if (p / "Lecture" / "notebooks" / "viz.py").exists())
sys.path.insert(0, str(NB))
from viz import chart, draw_graph, draw_mesh, draw_plan, draw_points, ramp, row, view3d

from compas.colors import Color
from compas.geometry import Line
from compas.tolerance import Tolerance
from compas_timber.connections import (JointTopology, LButtJoint, LMiterJoint, TButtJoint,
                                       TTenonMortiseJoint, XLapJoint)
from compas_timber.elements import Beam
from compas_timber.model import TimberModel

## Step 1 — a beam is a centreline and a cross-section
You draw a line; the beam adds what a line lacks: a width, a height, and a
`z_vector` that says which way the section is turned.

In [ ]:
beam = Beam.from_centerline(Line([0, 0, 0], [2000, 0, 0]), width=60, height=120, z_vector=[0, 1, 0])
print("length", beam.length, "| width", beam.width, "| height", beam.height)
print("frame: ", beam.frame)
print("blank: ", beam.blank)

The **blank** is the piece of timber you would take off the shelf: a box.
It is all the geometry this notebook needs. (`beam.geometry`, the blank with
the joints cut into it, needs Rhino.)

## Step 2 — a model, and a helper to make one
A `TimberModel` holds the beams. Tell it the unit, so that its tolerance
means something.

In [ ]:
def model_of(lines, width=60, height=120):
    """A TimberModel from {name: Line}."""
    model = TimberModel(tolerance=Tolerance(unit="MM", absolute=1e-3))
    for name, line in lines.items():
        beam = Beam.from_centerline(line, width, height, z_vector=[0, 1, 0])
        beam.name = name
        model.add_element(beam)
    return model


def show(model):
    palette = ["#1c7ed6", "#e8590c", "#2f9e44", "#7048e8"]
    view3d(*[(beam.blank, {"color": Color.from_hex(palette[i % 4])}) for i, beam in enumerate(model.beams)])


corner = model_of({"sill": Line([0, 0, 0], [2000, 0, 0]), "post": Line([0, 0, 0], [0, 0, 1500])})
print([beam.name for beam in corner.beams])
show(corner)

## Step 3 — three ways for two beams to meet
`connect_adjacent_beams()` looks at the centrelines and proposes **joint
candidates**, each with a *topology*:

- **L**: two ends meet (a corner)
- **T**: one end meets the side of the other
- **X**: the two cross

Four pairs of lines, same sill, the post in a different place each time.

In [ ]:
NAMES = {JointTopology.TOPO_L: "L", JointTopology.TOPO_T: "T", JointTopology.TOPO_X: "X"}
sill = Line([0, 0, 0], [2000, 0, 0])
cases = {
    "post at the end": Line([0, 0, 0], [0, 0, 1500]),
    "post in the middle": Line([1000, 0, 0], [1000, 0, 1500]),
    "post through the sill": Line([1000, 0, -700], [1000, 0, 700]),
    "post 500 mm away": Line([2500, 0, 0], [2500, 0, 1500]),
}
for name, post in cases.items():
    model = model_of({"sill": sill, "post": post})
    model.connect_adjacent_beams()
    found = [NAMES[c.topology] for c in model.joint_candidates]
    print(f"{name:22s} -> {found or 'no candidate'}")

## Step 4 — the model is a graph
Every model carries a COMPAS `Graph`: a node per beam, an edge per
candidate or joint. This is the *element graph* of `03_two_graphs_of_one_frame`, built for
you.

In [ ]:
corner.connect_adjacent_beams()
graph = corner.graph
print("nodes:", list(graph.nodes()), "| edges:", list(graph.edges()))
for candidate in corner.joint_candidates:
    print("candidate:", NAMES[candidate.topology], "between", [e.name for e in candidate.elements],
          "at", candidate.location)

## Step 5 — a joint is a decision on an edge
A candidate only says *these two meet*. Choosing **how** is design. Take
the corner and make it twice: a butt joint, and a miter. Then ask each beam
what was done to it: its `features` are the machining, and its
`blank_length` is how long a piece you must order.

In [ ]:
def describe(model):
    for beam in model.beams:
        cuts = [type(f).__name__.replace("Proxy", "") for f in beam.features]
        print(f"   {beam.name:5s} centreline {beam.length:6.0f}  blank {beam.blank_length:6.0f}  machining {cuts}")


for joint in (LButtJoint, LMiterJoint):
    model = model_of({"sill": Line([0, 0, 0], [2000, 0, 0]), "post": Line([0, 0, 0], [0, 0, 1500])})
    a, b = model.beams
    joint.create(model, a, b)
    errors = model.process_joinery()
    print(f"{joint.__name__}: {len(errors)} errors")
    describe(model)

Both beams get one cut either way. The difference is in the blanks: for the
butt joint only the post is extended past the corner, for the miter both
are. The centrelines never changed. **The joint decides the cutting list.**

### A T joint has an order
One beam ends (the *main* beam), the other runs through (the *cross*
beam). `TButtJoint.create(model, main, cross)` cuts the main beam. Swap the
arguments and it cuts the wrong one, without any error.

In [ ]:
tee = {"sill": Line([0, 0, 0], [2000, 0, 0]), "post": Line([1000, 0, 0], [1000, 0, 1500])}
for order in (("post", "sill"), ("sill", "post")):
    model = model_of(tee)
    beams = {beam.name: beam for beam in model.beams}
    TButtJoint.create(model, beams[order[0]], beams[order[1]])
    model.process_joinery()
    print(f"main = {order[0]}, cross = {order[1]}")
    describe(model)

In the second result the sill is the beam that gets cut, and the post is
left as it was: nonsense, delivered with zero errors. The stud wall in the
next notebook checks which beam ends at which before it creates a joint.

### A different joint, different machining
Same two beams, joined by a mortise and tenon instead.

In [ ]:
model = model_of(tee)
beams = {beam.name: beam for beam in model.beams}
TTenonMortiseJoint.create(model, beams["post"], beams["sill"])
model.process_joinery()
describe(model)

## Step 6 — where "headless" stops
Crossing beams want a lap joint: half of each cut away. Working that out
needs solid geometry that compas_timber takes from Rhino. Outside Rhino it
says so:

In [ ]:
model = model_of({"sill": sill, "post": cases["post through the sill"]})
a, b = model.beams
try:
    XLapJoint.create(model, a, b)
    model.process_joinery()
    print("lap joint made")
except Exception as error:
    print("XLapJoint ->", type(error).__name__)
    print("butt, miter and tenon joints work anywhere; lap, notch and dovetail joints need Rhino")

## Step 7 — the file a machine reads
BTLx is the exchange format of timber CNC machines: one `Part` per beam,
with its blank size and its list of `Processings`. Write it, then **read it
back**, because the file is the product and it is the file you should check.

In [ ]:
import xml.etree.ElementTree as ET

from compas_timber.fabrication import BTLxWriter

model = model_of(tee)
beams = {beam.name: beam for beam in model.beams}
TTenonMortiseJoint.create(model, beams["post"], beams["sill"])
model.process_joinery()

xml = BTLxWriter().model_to_xml(model)
print(len(xml), "characters of XML")
for element in ET.fromstring(xml).iter():
    tag = element.tag.split("}")[-1]
    if tag == "Part":
        print(f"Part {element.get('Annotation'):5s} {element.get('Length')} x {element.get('Width')} x {element.get('Height')}")
    elif tag == "Processings":
        print("   processings:", [child.tag.split("}")[-1] for child in element])

## Your turn
1. In Step 3, move the post to `x = 1990`. Is it an L or a T? Then try
   `connect_adjacent_beams(max_distance=50)`.
2. Make the corner with `LButtJoint.create(model, b, a)` (swapped). Which
   blank grows now? What does the order mean for an L joint?
3. Change the section to 80 by 160 in `model_of`. Which numbers in Step 5
   change, and by how much?
4. Add a third beam (a brace from the sill to the post). How many
   candidates does the model find, and of which topology?